
# Yelp Dataset — Exploration + Parquet/DuckDB

Tento notebook:
- převádí JSONL → **Parquet** (particionovaně) pomocí **DuckDB**,
- dělá rychlé **ad‑hoc SQL** dotazy nad Parquetem,
- sestaví **interactions.parquet** (user_id, business_id, ts, stars, implicit=1),
- ukáže **časový split** pro trénink/validaci,
- nechá skeleton pro ELSA + SAE (budeme napojovat podle tvé volby knihoven).


In [10]:

from pathlib import Path
import duckdb, pandas as pd
import numpy as np
import json, os, itertools
from datetime import datetime

# ---- Cesty k datům ----
DATA_DIR = Path("./Yelp JSON/yelp_dataset")          # změň dle sebe
PARQUET_DIR = Path("./Yelp JSON/yelp_parquet")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect("yelp.duckdb")

FILES = {
    "business": DATA_DIR / "yelp_academic_dataset_business.json",
    "review":   DATA_DIR / "yelp_academic_dataset_review.json",
    "user":     DATA_DIR / "yelp_academic_dataset_user.json",
    "tip":      DATA_DIR / "yelp_academic_dataset_tip.json",
    "checkin":  DATA_DIR / "yelp_academic_dataset_checkin.json",
}
FILES


{'business': WindowsPath('Yelp JSON/yelp_dataset/yelp_academic_dataset_business.json'),
 'review': WindowsPath('Yelp JSON/yelp_dataset/yelp_academic_dataset_review.json'),
 'user': WindowsPath('Yelp JSON/yelp_dataset/yelp_academic_dataset_user.json'),
 'tip': WindowsPath('Yelp JSON/yelp_dataset/yelp_academic_dataset_tip.json'),
 'checkin': WindowsPath('Yelp JSON/yelp_dataset/yelp_academic_dataset_checkin.json')}

## 1) JSON → Parquet pomocí DuckDB

In [12]:

# Business (partitions by state)
if FILES["business"].exists():
    con.execute(f"""
    COPY (
      SELECT * FROM read_json_auto('{FILES["business"]}', format='auto')
    ) TO '{PARQUET_DIR/'business'}' (
      FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (state)
    );
    """)
    print("business → parquet OK")
else:
    print("business.json nenalezen")

# Review (partitions by year)
if FILES["review"].exists():
    con.execute(f"""
    COPY (
      SELECT *, CAST(strftime(date, '%Y') AS INTEGER) AS year
      FROM read_json_auto('{FILES["review"]}', format='auto')
    ) TO '{PARQUET_DIR/'review'}' (
      FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (year)
    );
    """)
    print("review → parquet OK")
else:
    print("review.json nenalezen")

# User (no partitions, můžeš přidat podle yelping_since)
if FILES["user"].exists():
    con.execute(f"""
    COPY (
      SELECT * FROM read_json_auto('{FILES["user"]}', format='auto')
    ) TO '{PARQUET_DIR/'user.parquet'}' (
      FORMAT PARQUET, COMPRESSION ZSTD
    );
    """)
    print("user → parquet OK")
else:
    print("user.json nenalezen")

# Tip (volitelné)
if FILES["tip"].exists():
    con.execute(f"""
    COPY (
      SELECT * FROM read_json_auto('{FILES["tip"]}', format='auto')
    ) TO '{PARQUET_DIR/'tip.parquet'}' (
      FORMAT PARQUET, COMPRESSION ZSTD
    );
    """)
    print("tip → parquet OK")

# Checkin (volitelné – může se hodit na implicitní interakce)
if FILES["checkin"].exists():
    con.execute(f"""
    COPY (
      SELECT * FROM read_json_auto('{FILES["checkin"]}', format='auto')
    ) TO '{PARQUET_DIR/'checkin.parquet'}' (
      FORMAT PARQUET, COMPRESSION ZSTD
    );
    """)
    print("checkin → parquet OK")


business → parquet OK
review → parquet OK
user → parquet OK
tip → parquet OK
checkin → parquet OK


## 2) Rychlé SQL nad Parquetem (bez loadu do RAM)

In [13]:

# Příklad: top města podle počtu podniků a průměrných hvězd
q = con.execute(f"""
SELECT city, state, COUNT(*) AS n, AVG(CAST(stars AS DOUBLE)) AS avg_stars
FROM read_parquet('{PARQUET_DIR/'business/*/*.parquet'}')
GROUP BY city, state
ORDER BY n DESC
LIMIT 20
""").fetchdf()
q


,city,state,n,avg_stars
0,Philadelphia,PA,14567,3.623224
1,Tucson,AZ,9249,3.594767
2,Tampa,FL,9048,3.583057
3,Indianapolis,IN,7540,3.579708
4,Nashville,TN,6968,3.637629
5,New Orleans,LA,6208,3.822890
6,Reno,NV,5932,3.760958
7,Edmonton,AB,5054,3.439058
8,Saint Louis,MO,4827,3.594054
9,Santa Barbara,CA,3829,4.051449


## 3) Vytvoření interactions.parquet (implicit = 1 z reviews)

In [15]:

# Z recenzí vytvoříme interakce (user_id, business_id, timestamp, stars, implicit=1)
# Data se čtou přímo z Parquetu, můžeš přidat filtry (rok, stát) pro menší subsety.
interactions_path = PARQUET_DIR / "interactions.parquet"

con.execute(f"""
COPY (
  SELECT
    user_id,
    business_id,
    TRY_CAST(stars AS DOUBLE) AS stars,
    TRY_CAST(epoch_ms(date) AS BIGINT) AS ts,
    1 AS implicit
  FROM read_parquet('{PARQUET_DIR/'review/year=*/data-*.parquet'}')
  WHERE user_id IS NOT NULL AND business_id IS NOT NULL
) TO '{interactions_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")
print("interactions.parquet vytvořen:", interactions_path.exists())


IOException: IO Error: No files found that match the pattern "Yelp JSON\yelp_parquet\review\year=*\data-*.parquet"

LINE 9:   FROM read_parquet('Yelp JSON\yelp_parquet\review\year=*\data...
               ^

## 4) Časový split (train/valid/test)

In [16]:

import pandas as pd

def temporal_split(parquet_path: Path, valid_ratio=0.1, test_ratio=0.1):
    df = pd.read_parquet(parquet_path, columns=["user_id","business_id","ts","stars","implicit"])
    df = df.dropna(subset=["user_id","business_id","ts"]).sort_values("ts")
    tmin, tmax = int(df.ts.min()), int(df.ts.max())
    span = tmax - tmin
    valid_cut = tmin + int((1 - (valid_ratio + test_ratio)) * span)
    test_cut  = tmin + int((1 - test_ratio) * span)
    train = df[df.ts <= valid_cut]
    valid = df[(df.ts > valid_cut) & (df.ts <= test_cut)]
    test  = df[df.ts > test_cut]
    return train, valid, test

train, valid, test = temporal_split(interactions_path, 0.1, 0.1)
[len(train), len(valid), len(test)], [train.ts.min(), train.ts.max(), test.ts.min(), test.ts.max()]


FileNotFoundError: [Errno 2] No such file or directory: 'Yelp JSON\\yelp_parquet\\interactions.parquet'

## 5) Příprava uživatelského a item indexu (ID mapping)

In [ ]:

# Mapování textových ID na integer indexy pro matice
def build_id_map(series):
    uniq = series.drop_duplicates().reset_index(drop=True)
    return pd.Series(index=uniq.values, data=np.arange(len(uniq)), name="idx")

uid_map = build_id_map(train["user_id"])
iid_map = build_id_map(train["business_id"])

def remap(df):
    df = df.copy()
    df["u"] = df["user_id"].map(uid_map)
    df["i"] = df["business_id"].map(iid_map)
    return df.dropna(subset=["u","i"]).astype({"u": int, "i": int})

train_m = remap(train); valid_m = remap(valid); test_m = remap(test)
train_m.head()


## 6) ELSA + SAE skeleton

In [ ]:

# --- ELSA skeleton ---
# ELSA je lineární model pracující s item reprezentacemi a podobnostmi.
# Zde necháme placeholder rozhraní; implementaci/volbu knihovny doplníme dle tvé preference.

class ELSAModel:
    def __init__(self, n_items: int, reg: float = 1e-4):
        self.n_items = n_items
        self.reg = reg
        self.item_factors = None  # např. R^k pro itemy (může vzniknout z EASE/ALS/…)

    def fit(self, interactions_df):
        # TODO: napojit na zvolený pretraining (EASE/ALS) → získat item_factors
        # a vytvořit podobnostní strukturu pro rychlé dotazy.
        pass

    def user_embed(self, user_history_indices):
        # TODO: složit embed z historie (agregace/federated attention)
        pass

    def recommend(self, user_history_indices, topk=50, filter_seen=True):
        # TODO: skórování přes podobnosti
        pass

# --- SAE skeleton (Top-K SAE) ---
# SAE bude trénovat nad uživatelskými nebo item embeddingy z ELSA a poskytne sparse interpretable neurony.
class TopKSAE:
    def __init__(self, input_dim: int, hidden_dim: int, k: int = 32, l1: float = 1e-3):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.k = k
        self.l1 = l1
        # TODO: weights init

    def fit(self, X):
        # TODO: trénink se sparsitou (Top-K aktivace / thresholding)
        pass

    def encode(self, X):
        # TODO: návrat sparse kódů
        pass



## 7) Další kroky
- Přidat filtry (země/region, roky) do `COPY … PARTITION_BY` pro menší subsety na rychlé iterace.
- Rozhodnout o pretrainingu pro ELSA (EASE/ALS/Item2Vec) a napojit `item_factors`.
- Přidat metriky (Recall@K, NDCG@K) a offline evaluaci na `valid/test`.
- Přidat export **explainers** z SAE (pojmenování neuronů ze slov v kategoriích/business attributes).
